# Phase 0 / Lesson 3 — GPU Setup & Benchmark (Colab)

Run this on Google Colab with a **T4 GPU** runtime.

**Before running:** Runtime > Change runtime type > Hardware accelerator = **T4 GPU** > Save.

Then: Runtime > Run all.

## 1. Verify the GPU (Exercise: confirm hardware)

In [ ]:
!nvidia-smi

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version:   {torch.version.cuda}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"Memory:         {props.total_memory / 1e9:.1f} GB")
    print(f"Compute cap.:   {props.major}.{props.minor}")
else:
    print("\nNo GPU detected. Set Runtime > Change runtime type > T4 GPU, then re-run.")

: 

## 2. CPU vs GPU benchmark (Exercise 1)

Note the warm-up matmul and `torch.cuda.synchronize()` — without them the GPU timing is wrong because CUDA kernels launch asynchronously.

In [ ]:
import torch, time

size = 5000
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

# CPU (warm-up then average of 3)
_ = a_cpu @ b_cpu
start = time.time()
for _ in range(3):
    _ = a_cpu @ b_cpu
cpu_time = (time.time() - start) / 3
print(f"CPU: {cpu_time:.3f}s")

if torch.cuda.is_available():
    a_gpu = a_cpu.to("cuda")
    b_gpu = b_cpu.to("cuda")
    # warm-up (first CUDA call compiles/loads kernels)
    _ = a_gpu @ b_gpu
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(3):
        _ = a_gpu @ b_gpu
    torch.cuda.synchronize()
    gpu_time = (time.time() - start) / 3
    print(f"GPU: {gpu_time:.3f}s")
    print(f"Speedup: {cpu_time / gpu_time:.0f}x")
else:
    print("No GPU - speedup section skipped.")

## 3. How big a model fits? (Exercise 3)

Rule of thumb: fp16 = **2 bytes per parameter**. This is weights only — training also needs gradients + optimizer state (~3-4x more).

In [ ]:
import torch

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    vram_gb = 16  # T4 default

params_fp16_b = vram_gb * 1e9 / 2 / 1e9
print(f"VRAM: {vram_gb:.1f} GB")
print(f"Max model (fp16, inference, weights only): ~{params_fp16_b:.0f}B params")
print(f"Comfortable to TRAIN (~4x overhead):       ~{params_fp16_b/4:.1f}B params")